In [ ]:
import pandas as pd
import re

import re
import pandas as pd

def anonimizar_texto(texto):
    if pd.isna(texto):
        return texto

    texto = str(texto)

    # Corrigir ausência de espaços
    texto = re.sub(r'(OAB\s*[A-Z]{2}\d+)(?=[A-Z])', r'\1 ', texto)

    # Mascarar CPF
    texto = re.sub(r'\d{3}\.\d{3}\.\d{3}-\d{2}', 'XXX.XXX.XXX-XX', texto)

    mapa = {}
    contador = 1

    def get_id(nome):
        nonlocal contador
        nome = nome.strip()

        if nome not in mapa:
            mapa[nome] = f' PESSOA_{contador}'
            contador += 1

        return mapa[nome]

    # Identidicar Nome + CPF
    texto = re.sub(
        r'([A-ZÁÉÍÓÚÃÕÇ\s]+?)\s*-\s*CPF:',
        lambda m: f"{get_id(m.group(1))} - CPF:",
        texto
    )

    # Identificar Nome + "registrado(a) civilmente como"
    texto = re.sub(
        r'([A-ZÁÉÍÓÚÃÕÇ\s]+?)\s+registrado\(a\)\s+civilmente\s+como',
        lambda m: f"{get_id(m.group(1))} registrado(a) civilmente como",
        texto,
        flags=re.IGNORECASE
    )

    # Identificar Nome + OAB
    texto = re.sub(
        r'([A-ZÁÉÍÓÚÃÕÇ\s]+?)\s*-\s*OAB\s*([A-Z]{2}\d+)',
        lambda m: f"{get_id(m.group(1))} - OAB XXX",
        texto
    )

    # Garantia extra: qualquer OAB restante → anonimiza
    texto = re.sub(r'OAB\s*[A-Z]{2}\d+', 'OAB XXX', texto)

    return texto

df = pd.read_excel('trf1_completos.xlsx', dtype={'Número do Processo': str})

colunas = ['Polo Ativo', 'Polo Passivo', 'Outros Interessados']

for col in colunas:
    if col in df.columns:
        df[col] = df[col].apply(anonimizar_texto)

df.to_excel('trf1_processos_anon.xlsx', index=False)

print("Arquivo anonimizado gerado com sucesso!")


Arquivo anonimizado gerado com sucesso!
